# Playground Series S6E11: Predicting Credit Card Fraud
## Quad-Model Architecture (GBDTs + PyTorch ResNet) & Interaction-Augmented Stacking

### Competition Overview & Mission
This notebook addresses Kaggle Playground Series (Season 6, Episode 11): **Predicting Credit Card Fraud**. 
Our objective is to predict the probability that a credit card transaction is fraudulent (`is_fraud = 1.0`) given a synthetically generated tabular dataset reflecting real-world financial transaction dynamics.

The competition is evaluated using the **Area Under the Receiver Operating Characteristic Curve (ROC-AUC)**. In heavily imbalanced fraud detection settings ($1.67\%$ positive class), ROC-AUC measures global ranking quality across all decision thresholds.

#### Core Methodological Pillars
1. **Exploration-Led Engineering**: All features originate from empirical data discoveries (compounding risk curves, CVV verification exhaustion, liquidity targets).
2. **Adaptive Bayesian Target Encoding**: Count-adaptive Empirical Bayes shrinkage preserving high-precision signals in rare fraud vectors.
3. **Shallow-Tree Regularization with Positive-Class Gradient Weighting**: Constraining tree complexity (`num_leaves=7` in LightGBM, `max_depth=3` in XGBoost, `depth=4` in CatBoost) with positive scale weighting (`scale_pos_weight=1.5`) to counter severe class imbalance.
4. **Quad-Model Foundation (GBDTs + PyTorch Tabular ResNet)**: Combining Leaf-Wise (LightGBM), Depth-Wise (XGBoost), Oblivious (CatBoost), and Non-Axis-Aligned Deep Learning (PyTorch ResNet with Entity Embeddings and in-memory weights).
5. **Multi-Strategy Ensembling & Interaction Stacking**: Direct rank-space Nelder-Mead optimization and 2nd-level context-conditioned meta-learning with deterministic strategy routing.
6. **Dual-Environment Portability**: Seamless execution across local Apple Silicon / CPU and Kaggle Kernel GPU environments.


# 1. Environment Detection & Hardware Configuration
---
To guarantee deterministic execution and reproducibility whether running in a local IDE (VS Code / Jupyter) or on Kaggle's GPU infrastructure, we configure thread bounds, filter non-critical runtime warnings, and inspect the runtime hardware accelerator.


In [ ]:
import copy
import os
import time
import warnings

# Enforce thread control across multi-core systems to prevent CPU thread thrashing
os.environ["OMP_NUM_THREADS"] = "4"
os.environ["MKL_NUM_THREADS"] = "4"
os.environ["OPENBLAS_NUM_THREADS"] = "4"
os.environ["VECLIB_MAXIMUM_THREADS"] = "4"
warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from sklearn.metrics import average_precision_score, precision_recall_curve, roc_auc_score, roc_curve
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler

GLOBAL_SEED = 42
np.random.seed(GLOBAL_SEED)
torch.manual_seed(GLOBAL_SEED)

PALETTE = {
    "fraud": "#E63946",       # Coral Red
    "legitimate": "#1D3557",  # Deep Navy
    "accent": "#457B9D",      # Slate Blue
    "neutral": "#A8DADC",     # Soft Cyan
    "highlight": "#F4A261",   # Amber
    "dark": "#2B2D42"
}
sns.set_theme(style="whitegrid", font="sans-serif")
plt.rcParams["figure.dpi"] = 120
plt.rcParams["axes.titlesize"] = 13
plt.rcParams["axes.labelsize"] = 10.5
plt.rcParams["xtick.labelsize"] = 9.5
plt.rcParams["ytick.labelsize"] = 9.5
plt.rcParams["legend.fontsize"] = 9.5

IS_KAGGLE = os.path.exists("/kaggle")
print(f"Runtime Environment: {'Kaggle Kernel' if IS_KAGGLE else 'Local Workstation'}")


## 1.1 Compute Accelerator & Cross-Validation Stratification Setup
We configure dynamic hardware execution:
- **Kaggle Kernel**: Automatically leverages CUDA GPU acceleration (`DEVICE = "gpu"`), executes full **10-Fold Stratified Cross-Validation**, and multi-seed ensembling (`SEEDS = [42, 2026]`).
- **Local Environment**: Defaults to 5-Fold validation on CPU / Apple Silicon MPS for rapid iterative verification.


In [ ]:
N_SPLITS = 10 if IS_KAGGLE else 5

if IS_KAGGLE:
    SEEDS = [42, 2026]
    DEVICE = "gpu"
    TORCH_DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    N_JOBS = -1
    print(f"[Environment: Kaggle Kernel] Accelerated by {TORCH_DEVICE} | CV: {N_SPLITS}-Fold | Seeds: {SEEDS}")
else:
    SEEDS = [42]
    DEVICE = "cpu"
    if torch.backends.mps.is_available():
        TORCH_DEVICE = torch.device("mps")
    else:
        TORCH_DEVICE = torch.device("cpu")
    N_JOBS = 4
    print(f"[Environment: Local System] Running on {TORCH_DEVICE} | CPU Workers: {N_JOBS} | CV: {N_SPLITS}-Fold | Seeds: {SEEDS}")


## 1.2 Adaptive Data Ingestion
The dataset consists of `train.csv` (699,960 records with binary label `is_fraud`), `test.csv` (299,984 records), and `sample_submission.csv`. 

Our data loader scans Kaggle input directories (`/kaggle/input/playground-series-s6e11`), parent directories, and local project folders systematically, ensuring zero-configuration execution in both settings.


In [ ]:
def load_data():
    """Robust data loader with automated fallback resolution for Kaggle and local environments."""
    train_file = "train.csv"
    test_file = "test.csv"
    sample_sub_file = "sample_submission.csv"

    data_dir = None

    if IS_KAGGLE:
        kaggle_candidates = [
            "/kaggle/input/playground-series-s6e11",
            "/kaggle/input/competitions/playground-series-s6e11",
        ]
        for candidate in kaggle_candidates:
            if os.path.exists(candidate) and os.path.exists(os.path.join(candidate, train_file)):
                data_dir = candidate
                break

        if data_dir is None and os.path.exists("/kaggle/input"):
            for root, _, files in os.walk("/kaggle/input"):
                if train_file in files:
                    data_dir = root
                    break
    else:
        local_candidates = [
            "data",
            "./data",
            "../data",
            os.path.join(os.getcwd(), "data"),
            "/Users/davidlangat/Documents/Github/Kaggle/playground_series/predicting_credit_card_fraud/data",
        ]
        for candidate in local_candidates:
            if os.path.exists(candidate) and os.path.exists(os.path.join(candidate, train_file)):
                data_dir = candidate
                break

    if data_dir is None:
        raise FileNotFoundError("Unable to locate train.csv in expected Kaggle or local directory candidates.")

    print(f"Data source resolved at: {data_dir}")
    t0 = time.time()
    train = pd.read_csv(os.path.join(data_dir, train_file))
    test = pd.read_csv(os.path.join(data_dir, test_file))
    submission = pd.read_csv(os.path.join(data_dir, sample_sub_file))
    load_time = time.time() - t0

    print(f"Loaded successfully in {load_time:.2f} seconds:")
    print(f" - Train Set:       {train.shape[0]:,} rows × {train.shape[1]} columns")
    print(f" - Test Set:        {test.shape[0]:,} rows × {test.shape[1]} columns")
    print(f" - Sample Sub Set:  {submission.shape[0]:,} rows × {submission.shape[1]} columns")

    return train, test, submission

train_df, test_df, submission_df = load_data()


# 2. Data-Driven Exploratory Data Analysis (EDA)
---
Rather than making assumptions, we perform an in-depth audit of the feature distributions, class balance, risk factor combinations, and vulnerability surfaces present in this dataset.

Every visual analysis in this section includes:
1. The quantitative data display.
2. A dedicated **Analytical Takeaway** detailing the mechanism behind the observed pattern.
3. The actionable **Downstream Modeling Implication**.


## 2.1 Data Integrity & Missing Value Audit
We evaluate whether any records contain missing, null, or corrupted values across both train and test partitions.


In [ ]:
def audit_data_integrity(df_train, df_test):
    train_nulls = df_train.isnull().sum()
    test_nulls = df_test.isnull().sum()

    audit = pd.DataFrame({
        "Train Dtype": df_train.dtypes,
        "Train Nulls": train_nulls,
        "Train Null %": (train_nulls / len(df_train) * 100).round(4),
        "Test Dtype": df_test.dtypes,
        "Test Nulls": test_nulls,
        "Test Null %": (test_nulls / len(df_test) * 100).round(4)
    })
    return audit

integrity_report = audit_data_integrity(train_df, test_df)
print(f"Total Columns Audited: {len(integrity_report)}")
print(f"Any Null Values in Train? {train_df.isnull().any().any()}")
print(f"Any Null Values in Test?  {test_df.isnull().any().any()}")
display(integrity_report)


> [!NOTE]
> **Analytical Takeaway & Modeling Implication:**
> Both `train.csv` and `test.csv` have **zero missing values** across all columns. The synthetic generation process has produced complete records without missing data artifacts.
> * **Modeling Implication:** No imputation strategies (such as iterative MICE or median filling) are required, preserving original feature distributions and avoiding imputation noise.


## 2.2 Target Distribution & Class Imbalance Analysis
In fraud detection, target class imbalance is typical. Quantifying the precise positive-class ratio is critical for selecting our validation split strategy, model loss functions, and evaluation thresholds.


In [ ]:
fraud_counts = train_df["is_fraud"].value_counts().sort_index()
fraud_pct = train_df["is_fraud"].value_counts(normalize=True).sort_index() * 100

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(["Legitimate (0)", "Fraud (1)"], fraud_counts.values, color=[PALETTE["legitimate"], PALETTE["fraud"]], width=0.45)

for bar, count, pct in zip(bars, fraud_counts.values, fraud_pct.values):
    yval = bar.get_height()
    ax.text(bar.get_x() + bar.get_width() / 2.0, yval + 10000, f"{count:,}\n({pct:.2f}%)", ha="center", va="bottom", fontsize=10, fontweight="bold")

ax.set_title("Ground Truth Class Distribution (is_fraud)", fontsize=13, fontweight="bold", pad=15)
ax.set_ylabel("Transaction Count", fontsize=11)
ax.set_ylim(0, 780000)
ax.grid(axis="y", linestyle="--", alpha=0.6)
plt.tight_layout()
plt.show()

print(f"Base Positive Rate (Fraud): {fraud_pct[1]:.4f}%")
print(f"Imbalance Ratio:            1 : {(fraud_counts[0] / fraud_counts[1]):.1f}")


> [!IMPORTANT]
> **Analytical Takeaway & Modeling Implication:**
> The dataset has a **$1.67\%$ base fraud rate** ($11,690$ fraudulent cases out of $699,960$ total observations), representing an imbalance ratio of **$59 : 1$**.
> * **Modeling Implication 1:** Standard K-Fold splits will introduce fold-to-fold class variance. Strict **Stratified K-Fold Cross-Validation (`StratifiedKFold`)** is essential to lock identical positive-class prevalence across all folds.
> * **Modeling Implication 2:** Accuracy is an invalid metric. Our models will optimize probabilities directly for **ROC-AUC** and monitor **Precision-Recall AUC (PR-AUC)**.


## 2.3 Compounding Anomaly Risk Curve Analysis
The dataset contains multiple binary indicators of anomalous transaction behavior:
`is_foreign_transaction`, `is_new_merchant`, `used_vpn`, `ip_country_mismatch`, `billing_shipping_mismatch`, `is_ai_generated_scam_attempt`.

We analyze whether fraud probability scales linearly or exponentially as these flags compound simultaneously.


In [ ]:
risk_flags = [
    "is_foreign_transaction",
    "is_new_merchant",
    "used_vpn",
    "ip_country_mismatch",
    "billing_shipping_mismatch",
    "is_ai_generated_scam_attempt"
]

train_df["temp_risk_count"] = train_df[risk_flags].sum(axis=1)
risk_curve = train_df.groupby("temp_risk_count")["is_fraud"].agg(["count", "mean", "sum"]).reset_index()
risk_curve["fraud_rate_pct"] = risk_curve["mean"] * 100

fig, ax1 = plt.subplots(figsize=(9, 4.5))

color1 = PALETTE["accent"]
ax1.set_xlabel("Number of Simultaneous Anomaly Flags Present", fontsize=11, fontweight="bold")
ax1.set_ylabel("Total Transactions (Volume)", color=color1, fontsize=11, fontweight="bold")
b1 = ax1.bar(risk_curve["temp_risk_count"], risk_curve["count"], color=color1, alpha=0.45, width=0.5, label="Transaction Volume")
ax1.tick_params(axis="y", labelcolor=color1)
ax1.set_yscale("log")

ax2 = ax1.twinx()
color2 = PALETTE["fraud"]
ax2.set_ylabel("Empirical Fraud Rate (%)", color=color2, fontsize=11, fontweight="bold")
l1 = ax2.plot(risk_curve["temp_risk_count"], risk_curve["fraud_rate_pct"], color=color2, linewidth=3, marker="o", markersize=8, label="Fraud Rate %")
ax2.tick_params(axis="y", labelcolor=color2)
ax2.grid(False)

for _, row in risk_curve.iterrows():
    ax2.annotate(
        f"{row['fraud_rate_pct']:.2f}%",
        (row["temp_risk_count"], row["fraud_rate_pct"]),
        textcoords="offset points",
        xytext=(0, 10),
        ha="center",
        fontsize=9.5,
        fontweight="bold",
        color=color2
    )

train_df.drop(columns=["temp_risk_count"], inplace=True)
plt.title("Compounding Anomaly Risk Curve: Exponential Escalation", fontsize=13, fontweight="bold", pad=15)
plt.tight_layout()
plt.show()


> [!NOTE]
> **Analytical Takeaway & Modeling Implication:**
> Anomaly flags exhibit a strong compounding effect. While transactions with **0 flags** have a fraud rate of only **$0.42\%$**, transactions with **3 flags** exceed **$12.5\%$**, and transactions with **4+ flags** exceed **$38\%$**.
> * **Modeling Implication:** A composite risk indicator (`composite_risk_score = sum(risk_flags)`) provides tree splitters and neural networks with an immediate monotonic gradient of threat severity.


## 2.4 CVV Retry Exhaustion & Verification Failure Dynamics
In genuine cardholder transactions, entering an incorrect CVV is rare. Repeated retries (`cvv_retry_count`) indicate either automated brute-force attacks or compromised credentials.


In [ ]:
cvv_analysis = train_df.groupby("cvv_retry_count")["is_fraud"].agg(["count", "mean"]).reset_index()
cvv_analysis["fraud_rate_pct"] = cvv_analysis["mean"] * 100

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(cvv_analysis["cvv_retry_count"].astype(str), cvv_analysis["fraud_rate_pct"], color=PALETTE["fraud"], width=0.5)

for bar, rate, count in zip(bars, cvv_analysis["fraud_rate_pct"], cvv_analysis["count"]):
    ax.text(
        bar.get_x() + bar.get_width() / 2.0,
        bar.get_height() + 0.3,
        f"{rate:.2f}%\n(N={count:,})",
        ha="center",
        va="bottom",
        fontsize=9.5,
        fontweight="bold"
    )

ax.set_title("Fraud Prevalence vs. CVV Verification Retry Count", fontsize=13, fontweight="bold", pad=15)
ax.set_xlabel("CVV Retry Count", fontsize=11)
ax.set_ylabel("Empirical Fraud Rate (%)", fontsize=11)
ax.set_ylim(0, max(cvv_analysis["fraud_rate_pct"]) * 1.25)
ax.grid(axis="y", linestyle="--", alpha=0.6)
plt.tight_layout()
plt.show()


> [!NOTE]
> **Analytical Takeaway & Modeling Implication:**
> When `cvv_retry_count == 0`, fraud prevalence is minimal. However, as retries reach **2, 3, or 4**, the probability of fraud escalates dramatically.
> * **Modeling Implication:** We will construct non-linear interaction terms coupling CVV retries with velocity and merchant risk (`cvv_retry_count * velocity_score`, `cvv_retry_count * merchant_risk_score`).


## 2.5 Categorical Vulnerability Surface & Authentication Channel Gap
Fraudsters preferentially route stolen funds through specific channels and merchant categories where cash conversion is immediate and physical identity verification is weak.


In [ ]:
pivot_fraud = train_df.pivot_table(
    index="merchant_category",
    columns="auth_method",
    values="is_fraud",
    aggfunc="mean"
) * 100

fig, ax = plt.subplots(figsize=(10, 5.5))
sns.heatmap(
    pivot_fraud,
    annot=True,
    fmt=".2f",
    cmap="YlOrRd",
    cbar_kws={"label": "Empirical Fraud Rate (%)"},
    ax=ax,
    linewidths=0.5
)
ax.set_title("Vulnerability Matrix: Merchant Category vs. Authentication Method", fontsize=13, fontweight="bold", pad=15)
ax.set_xlabel("Authentication Method", fontsize=11)
ax.set_ylabel("Merchant Category", fontsize=11)
plt.tight_layout()
plt.show()


> [!NOTE]
> **Analytical Takeaway & Modeling Implication:**
> Transactions lacking step-up authentication (`No Authentication`) targeting high-liquidity merchant categories (**Crypto Exchange**, **Gift Cards**, **Online Gaming**) exhibit fraud rates exceeding **$15-22\%$**, while **Biometric** or **SMS OTP** transactions in **Grocery Stores** have fraud rates below **$0.3\%$**.
> * **Modeling Implication:** High-order interaction tokens (`merchant_category + "_" + auth_method`) combined with adaptive target encoding will encode these specific attack channels.


## 2.6 Digit-Level Artifacts & Fractional Cents Analysis
Synthetic tabular data generators (CTGAN, CopulaGAN) often introduce subtle distribution artifacts in the floating-point decimal expansions and rounding behaviors of transaction amounts.


In [ ]:
train_df["temp_cents"] = np.round((train_df["amount_usd"] * 100) % 100).astype(int)
cents_stats = train_df.groupby("temp_cents")["is_fraud"].agg(["count", "mean"]).reset_index()
cents_stats["fraud_rate_pct"] = cents_stats["mean"] * 100

fig, ax = plt.subplots(figsize=(12, 4))
ax.scatter(cents_stats["temp_cents"], cents_stats["fraud_rate_pct"], color=PALETTE["accent"], alpha=0.8, s=35)
ax.axhline(train_df["is_fraud"].mean() * 100, color=PALETTE["fraud"], linestyle="--", label="Global Mean (1.67%)")

top_cents = cents_stats.sort_values(by="fraud_rate_pct", ascending=False).head(5)
for _, r in top_cents.iterrows():
    ax.annotate(f".{int(r['temp_cents']):02d}: {r['fraud_rate_pct']:.2f}%", 
                (r["temp_cents"], r["fraud_rate_pct"]),
                textcoords="offset points", xytext=(0, 8), ha="center", fontsize=8.5, fontweight="bold")

ax.set_title("Empirical Fraud Rate across Fractional Cents (.00 to .99)", fontsize=13, fontweight="bold")
ax.set_xlabel("Cents Value (amount_usd * 100 % 100)", fontsize=11)
ax.set_ylabel("Fraud Rate (%)", fontsize=11)
ax.legend(loc="upper right")
ax.grid(True, linestyle="--", alpha=0.6)
train_df.drop(columns=["temp_cents"], inplace=True)
plt.tight_layout()
plt.show()


> [!NOTE]
> **Analytical Takeaway & Modeling Implication:**
> Specific fractional cents values (e.g. `.00`, `.49`, `.03`) exhibit fraud rates over $2.3-2.6\%$, whereas other values (e.g. `.64`, `.24`) exhibit fraud rates under $1.1\%$ (>2x difference).
> * **Modeling Implication:** Extracting exact cents (`amount_cents`), round dollar flags (`is_round_dollar`), and decimal residue features isolates these synthetic generator artifacts.


# 3. Targeted Feature Engineering Pipeline
---
Guided by our empirical EDA, we construct a domain-specific feature pipeline:
1. **Compounding Anomaly Aggregations**: `composite_risk_score`, `has_multiple_risk_flags`, `is_zero_risk`, `cross_border_evasion`.
2. **Micro-Cluster Scam Flags**: Explicit detection of high-risk liquidity channels with failed authentication (`unauth_retry_scam`, `extreme_velocity_evasion`).
3. **Fractional & Digit Artifacts**: `amount_cents`, `is_round_dollar`, `is_99_cents`, `is_50_cents`, decimal residue features.
4. **Behavioral CVV & Velocity Indicators**: `cvv_retry_present`, `cvv_x_velocity`, `cvv_x_merchant_risk`.
5. **Contextual Group-by Z-Scores & Train-Test Aligned Decile Bins**: Category-normalized amount deviations and quantile-locked cohort velocity differences.
6. **Adaptive Empirical Bayes Target Encoding**: Dynamic shrinkage that preserves sharp positive rates on targeted micro-clusters without over-smoothing.


In [ ]:
def engineer_features(df, cat_medians=None, cat_amount_stats=None, age_velocity_stats=None, age_bins=None):
    """Vectorized feature engineering incorporating behavioral composites, micro-clusters, and quantile-aligned cohorts."""
    df = df.copy()

    # 1. Anomaly & Security Flag Aggregations
    risk_flags = [
        "is_foreign_transaction",
        "is_new_merchant",
        "used_vpn",
        "ip_country_mismatch",
        "billing_shipping_mismatch",
        "is_ai_generated_scam_attempt"
    ]
    df["composite_risk_score"] = df[risk_flags].sum(axis=1)
    df["has_multiple_risk_flags"] = (df["composite_risk_score"] >= 2).astype(int)
    df["is_zero_risk"] = (df["composite_risk_score"] == 0).astype(int)
    df["address_geo_mismatch"] = (df["ip_country_mismatch"] | df["billing_shipping_mismatch"]).astype(int)
    df["cross_border_evasion"] = (
        df["is_foreign_transaction"].astype(bool) & 
        df["used_vpn"].astype(bool) & 
        df["ip_country_mismatch"].astype(bool)
    ).astype(int)

    # 2. Authentication Vulnerabilities & High-Risk Liquidity Targets
    df["is_unauthenticated"] = (df["auth_method"] == "No Authentication").astype(int)
    df["is_biometric_auth"] = (df["auth_method"] == "Biometric").astype(int)
    high_risk_cats = ["Gift Cards", "Crypto Exchange", "Gaming"]
    df["is_high_risk_category"] = df["merchant_category"].isin(high_risk_cats).astype(int)
    df["unauth_liquid_asset"] = (df["is_unauthenticated"] & df["is_high_risk_category"]).astype(int)

    # Micro-cluster scam combinations
    df["unauth_retry_scam"] = (
        (df["auth_method"] == "No Authentication") & 
        (df["cvv_retry_count"] > 0) & 
        (df["is_high_risk_category"] == 1)
    ).astype(int)
    df["extreme_velocity_evasion"] = (
        (df["velocity_score"] > 8.0) & 
        ((df["used_vpn"] == 1) | (df["is_foreign_transaction"] == 1))
    ).astype(int)

    # Categorical interaction tokens
    df["category_auth"] = df["merchant_category"].astype(str) + "_" + df["auth_method"].astype(str)
    df["channel_category"] = df["channel"].astype(str) + "_" + df["merchant_category"].astype(str)

    # 3. Behavioral CVV & Velocity Indicators
    df["cvv_retry_present"] = (df["cvv_retry_count"] > 0).astype(int)
    df["cvv_x_velocity"] = df["cvv_retry_count"] * df["velocity_score"]
    df["cvv_x_merchant_risk"] = df["cvv_retry_count"] * df["merchant_risk_score"]
    df["velocity_x_merchant_risk"] = df["velocity_score"] * df["merchant_risk_score"]

    # 4. Temporal & Velocity Dynamics
    df["velocity_to_hours_ratio"] = df["velocity_score"] / (df["hours_since_last_txn"] + 0.1)
    df["txn_burst_rate"] = df["txn_count_last_24h"] / (df["hours_since_last_txn"] + 0.5)

    # 5. Financial Exposure & Relative Pricing
    df["log_amount_usd"] = np.log1p(np.maximum(df["amount_usd"], 0))
    df["amount_to_balance_ratio"] = df["amount_usd"] / (df["account_balance_usd"] + 10.0)
    df["amount_x_velocity"] = df["amount_usd"] * df["velocity_score"]
    df["dispute_velocity_risk"] = df["prior_disputes"] * df["velocity_score"]

    # 6. Digit-Level Artifacts & Fractional Resizing
    df["amount_cents"] = np.round((df["amount_usd"] * 100) % 100).astype(int)
    df["is_round_dollar"] = (df["amount_cents"] == 0).astype(int)
    df["is_99_cents"] = (df["amount_cents"] == 99).astype(int)
    df["is_50_cents"] = (df["amount_cents"] == 50).astype(int)

    # Continuous decimal residues
    df["hours_decimal"] = df["hours_since_last_txn"] - np.floor(df["hours_since_last_txn"])
    df["distance_decimal"] = df["distance_from_home_km"] - np.floor(df["distance_from_home_km"])
    df["balance_decimal"] = df["account_balance_usd"] - np.floor(df["account_balance_usd"])

    # 7. Contextual Category Deviations & Z-Scores
    if cat_amount_stats is None:
        cat_amount_stats = df.groupby("merchant_category")["amount_usd"].agg(["mean", "std"]).to_dict("index")
    
    means = df["merchant_category"].map(lambda c: cat_amount_stats.get(c, {}).get("mean", 0.0))
    stds = df["merchant_category"].map(lambda c: cat_amount_stats.get(c, {}).get("std", 1.0))
    df["amount_cat_mean_diff"] = df["amount_usd"] - means
    df["amount_cat_zscore"] = df["amount_cat_mean_diff"] / (stds + 1e-5)

    if cat_medians is None:
        cat_medians = df.groupby("merchant_category")["amount_usd"].median().to_dict()
    df["amount_vs_cat_median_ratio"] = df["amount_usd"] / (df["merchant_category"].map(cat_medians) + 1.0)

    # Customer age cohort velocity deviation (Quantile-aligned across train and test)
    if age_bins is None:
        _, age_bins = pd.qcut(df["customer_age"], 10, retbins=True, duplicates="drop")
        age_bins[0] = -np.inf
        age_bins[-1] = np.inf
    df["age_decile"] = pd.cut(df["customer_age"], bins=age_bins, labels=False, include_lowest=True)
    if age_velocity_stats is None:
        age_velocity_stats = df.groupby("age_decile")["velocity_score"].mean().to_dict()
    df["velocity_age_cohort_diff"] = df["velocity_score"] - df["age_decile"].map(age_velocity_stats)
    df.drop(columns=["age_decile"], inplace=True)

    for col in risk_flags:
        df[col] = df[col].astype(int)

    return df, cat_medians, cat_amount_stats, age_velocity_stats, age_bins


In [ ]:
print("Executing enhanced feature engineering pipeline...")
t0_fe = time.time()

train_feat, train_medians, train_cat_stats, train_age_stats, train_age_bins = engineer_features(train_df)
test_feat, _, _, _, _ = engineer_features(
    test_df, 
    cat_medians=train_medians, 
    cat_amount_stats=train_cat_stats, 
    age_velocity_stats=train_age_stats,
    age_bins=train_age_bins
)

target_col = "is_fraud"
y_train = train_feat[target_col].values
X_train = train_feat.drop(columns=["id", target_col])
X_test = test_feat.drop(columns=["id"])
test_ids = test_feat["id"].values

# Frequency encodings
for col in ["merchant_category", "category_auth", "channel_category", "amount_cents"]:
    freq_map = X_train[col].value_counts(normalize=True).to_dict()
    X_train[f"{col}_freq"] = X_train[col].map(freq_map).fillna(0)
    X_test[f"{col}_freq"] = X_test[col].map(freq_map).fillna(0)

# Precompute Out-Of-Fold Adaptive Empirical Bayes Target Encoding
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=GLOBAL_SEED)
te_columns = ["category_auth", "channel_category", "merchant_category", "auth_method"]
global_mean = float(y_train.mean())

print(f"Precomputing {N_SPLITS}-Fold Adaptive Target Encoding for {te_columns}...")
for col in te_columns:
    train_arr = X_train[col].astype(str).values
    test_arr = X_test[col].astype(str).values

    train_te_col = np.zeros(len(X_train), dtype=np.float64)
    test_te_col = np.zeros(len(X_test), dtype=np.float64)

    for trn_idx, val_idx in skf.split(train_arr, y_train):
        trn_cats, trn_y = train_arr[trn_idx], y_train[trn_idx]
        df_trn = pd.DataFrame({"cat": trn_cats, "target": trn_y})
        stats = df_trn.groupby("cat")["target"].agg(["count", "mean"])
        counts = stats["count"].to_dict()
        means = stats["mean"].to_dict()

        # Adaptive Empirical Bayes Shrinkage (preserves sharp signals on rare clusters)
        smooth_dict = {}
        for k in counts:
            n_k = counts[k]
            m_k = min(30.0, max(8.0, np.sqrt(n_k)))
            smooth_dict[k] = (n_k * means[k] + m_k * global_mean) / (n_k + m_k)

        val_cats = train_arr[val_idx]
        train_te_col[val_idx] = [smooth_dict.get(c, global_mean) for c in val_cats]
        test_te_col += np.array([smooth_dict.get(c, global_mean) for c in test_arr]) / N_SPLITS

    X_train[f"{col}_TE"] = train_te_col
    X_test[f"{col}_TE"] = test_te_col

# Define categorical features list
categorical_features = [
    "merchant_category", "card_type", "auth_method", 
    "channel", "device_type", "category_auth", "channel_category"
]
for cat_col in categorical_features:
    X_train[cat_col] = X_train[cat_col].astype("category")
    X_test[cat_col] = X_test[cat_col].astype("category")

fe_duration = time.time() - t0_fe
print(f"Feature engineering completed in {fe_duration:.2f}s!")
print(f" - Training Matrix: {X_train.shape[0]:,} samples × {X_train.shape[1]} features")
print(f" - Testing Matrix:  {X_test.shape[0]:,} samples × {X_test.shape[1]} features")


# 4. Model 1: Shallow LightGBM (`num_leaves=7`)
---
### Why Shallow Trees Outperform Deep Trees on Imbalanced Data
In tabular data with extreme class imbalance ($1.67\%$ positive class), deep trees (`num_leaves=63`) overfit to sparse terminal leaves. 
By constraining tree complexity to **shallow trees (`num_leaves=7`)** with unweighted calibrated loss (`scale_pos_weight=1.0`), the model operates like a regularized generalized additive model (GAM), capturing robust additive signals (`cvv_x_merchant_risk`, `composite_risk_score`, `category_auth_TE`) without fitting positive-tail generator noise.


In [ ]:
import lightgbm as lgb

lgb_oof_probs = np.zeros(len(X_train))
lgb_test_probs = np.zeros(len(X_test))
lgb_fold_aucs = []
lgb_fold_praucs = []
lgb_feature_importances = pd.DataFrame()

lgb_params = {
    "objective": "binary",
    "metric": "auc",
    "boosting_type": "gbdt",
    "learning_rate": 0.05,
    "num_leaves": 7,              # Shallow tree regularization (monotonic gain on synthetic fraud)
    "feature_fraction": 0.80,
    "bagging_fraction": 0.85,
    "bagging_freq": 1,
    "min_child_samples": 50,
    "n_jobs": N_JOBS,
    "verbose": -1,
}

print(f"Starting Shallow LightGBM {N_SPLITS}-Fold Cross-Validation (num_leaves=7 | Seeds: {SEEDS})...")
t0_lgb = time.time()

for fold, (trn_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
    X_tr, y_tr = X_train.iloc[trn_idx], y_train[trn_idx]
    X_va, y_va = X_train.iloc[val_idx], y_train[val_idx]

    trn_data = lgb.Dataset(X_tr, label=y_tr)
    val_data = lgb.Dataset(X_va, label=y_va, reference=trn_data)

    fold_val_preds = np.zeros(len(val_idx))
    fold_test_preds = np.zeros(len(X_test))

    for seed in SEEDS:
        p = lgb_params.copy()
        p["random_state"] = seed
        model = lgb.train(
            p,
            trn_data,
            num_boost_round=2500,
            valid_sets=[val_data],
            callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)],
        )

        fold_val_preds += model.predict(X_va, num_iteration=model.best_iteration) / len(SEEDS)
        fold_test_preds += model.predict(X_test, num_iteration=model.best_iteration) / len(SEEDS)

        fold_importance = pd.DataFrame({
            "feature": X_tr.columns,
            "importance": model.feature_importance(importance_type="gain"),
            "fold": fold + 1
        })
        lgb_feature_importances = pd.concat([lgb_feature_importances, fold_importance], axis=0)

    lgb_oof_probs[val_idx] = fold_val_preds
    lgb_test_probs += fold_test_preds / N_SPLITS

    fold_auc = roc_auc_score(y_va, fold_val_preds)
    fold_prauc = average_precision_score(y_va, fold_val_preds)
    lgb_fold_aucs.append(fold_auc)
    lgb_fold_praucs.append(fold_prauc)
    print(f"LGB Fold {fold + 1:02d} | ROC-AUC: {fold_auc:.6f} | PR-AUC: {fold_prauc:.6f} | Best Iter: {model.best_iteration}")

lgb_training_duration = time.time() - t0_lgb
lgb_overall_auc = roc_auc_score(y_train, lgb_oof_probs)
lgb_overall_prauc = average_precision_score(y_train, lgb_oof_probs)

print("-" * 65)
print(f"Shallow LightGBM {N_SPLITS}-Fold Mean ROC-AUC: {np.mean(lgb_fold_aucs):.6f} (+/- {np.std(lgb_fold_aucs):.6f})")
print(f"Shallow LightGBM Overall OOF ROC-AUC:  {lgb_overall_auc:.6f}")
print(f"Shallow LightGBM Overall OOF PR-AUC:   {lgb_overall_prauc:.6f}")
print(f"Training completed in:                {lgb_training_duration:.2f} seconds")
print("-" * 65)


# 5. Model 2: Shallow XGBoost (`max_depth=3`)
---
Unlike LightGBM's leaf-wise strategy, XGBoost splits trees level-by-level (depth-wise). Setting `max_depth=3` limits trees to at most 8 terminal leaves, ensuring complementary structural diversity while retaining maximum regularization.


In [ ]:
import xgboost as xgb

xgb_oof_probs = np.zeros(len(X_train))
xgb_test_probs = np.zeros(len(X_test))
xgb_fold_aucs = []
xgb_fold_praucs = []
xgb_feature_importances = pd.DataFrame()

XGB_TREE_METHOD = "hist"
XGB_DEVICE = "cuda" if (IS_KAGGLE and DEVICE == "gpu") else "cpu"

xgb_params = {
    "objective": "binary:logistic",
    "eval_metric": "auc",
    "tree_method": XGB_TREE_METHOD,
    "device": XGB_DEVICE,
    "learning_rate": 0.05,
    "max_depth": 3,               # Constrained shallow depth (level-wise GAM)
    "colsample_bytree": 0.80,
    "subsample": 0.85,
    "min_child_weight": 50,
    "enable_categorical": True,
    "random_state": GLOBAL_SEED,
}

print(f"Starting Shallow XGBoost {N_SPLITS}-Fold Cross-Validation (Device: {XGB_DEVICE} | max_depth=3 | Seeds: {SEEDS})...")
t0_xgb = time.time()

for fold, (trn_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
    X_tr, y_tr = X_train.iloc[trn_idx], y_train[trn_idx]
    X_va, y_va = X_train.iloc[val_idx], y_train[val_idx]

    fold_val_preds = np.zeros(len(val_idx))
    fold_test_preds = np.zeros(len(X_test))

    for seed in SEEDS:
        p = xgb_params.copy()
        p["random_state"] = seed
        model = xgb.XGBClassifier(**p, n_estimators=2500, early_stopping_rounds=50)
        model.fit(
            X_tr, y_tr,
            eval_set=[(X_va, y_va)],
            verbose=False
        )

        fold_val_preds += model.predict_proba(X_va)[:, 1] / len(SEEDS)
        fold_test_preds += model.predict_proba(X_test)[:, 1] / len(SEEDS)

        fold_importance = pd.DataFrame({
            "feature": X_tr.columns,
            "importance": model.feature_importances_,
            "fold": fold + 1
        })
        xgb_feature_importances = pd.concat([xgb_feature_importances, fold_importance], axis=0)

    xgb_oof_probs[val_idx] = fold_val_preds
    xgb_test_probs += fold_test_preds / N_SPLITS

    fold_auc = roc_auc_score(y_va, fold_val_preds)
    fold_prauc = average_precision_score(y_va, fold_val_preds)
    xgb_fold_aucs.append(fold_auc)
    xgb_fold_praucs.append(fold_prauc)
    print(f"XGB Fold {fold + 1:02d} | ROC-AUC: {fold_auc:.6f} | PR-AUC: {fold_prauc:.6f} | Best Iter: {model.best_iteration}")

xgb_training_duration = time.time() - t0_xgb
xgb_overall_auc = roc_auc_score(y_train, xgb_oof_probs)
xgb_overall_prauc = average_precision_score(y_train, xgb_oof_probs)

print("-" * 65)
print(f"Shallow XGBoost {N_SPLITS}-Fold Mean ROC-AUC: {np.mean(xgb_fold_aucs):.6f} (+/- {np.std(xgb_fold_aucs):.6f})")
print(f"Shallow XGBoost Overall OOF ROC-AUC:  {xgb_overall_auc:.6f}")
print(f"Shallow XGBoost Overall OOF PR-AUC:   {xgb_overall_prauc:.6f}")
print(f"Training completed in:                {xgb_training_duration:.2f} seconds")
print("-" * 65)


# 6. Model 3: Shallow CatBoost (`depth=4`)
---
CatBoost introduces **symmetric / oblivious decision trees** that split data along balanced hyperplanes. By setting `depth=4`, CatBoost builds balanced 16-leaf trees that provide genuine structural diversity when combined with LightGBM and XGBoost.


In [ ]:
from catboost import CatBoostClassifier

cat_oof_probs = np.zeros(len(X_train))
cat_test_probs = np.zeros(len(X_test))
cat_fold_aucs = []
cat_fold_praucs = []
cat_feature_importances = pd.DataFrame()

CAT_TASK_TYPE = "GPU" if (IS_KAGGLE and DEVICE == "gpu") else "CPU"
CAT_ITERATIONS = 2000 if IS_KAGGLE else 600

cat_base_params = {
    "iterations": CAT_ITERATIONS,
    "learning_rate": 0.05,
    "depth": 4,               # Shallow symmetric oblivious depth (16 leaves)
    "l2_leaf_reg": 5.0,
    "eval_metric": "AUC",
    "task_type": CAT_TASK_TYPE,
    "early_stopping_rounds": 50,
    "metric_period": 10,      # Evaluate AUC periodically on host to suppress GPU warnings
    "verbose": False,
}

if CAT_TASK_TYPE == "CPU":
    cat_base_params["thread_count"] = N_JOBS

X_train_cat = X_train.copy()
X_test_cat = X_test.copy()
for c in categorical_features:
    X_train_cat[c] = X_train_cat[c].astype(str)
    X_test_cat[c] = X_test_cat[c].astype(str)

print(f"Starting Shallow CatBoost {N_SPLITS}-Fold Cross-Validation (Task: {CAT_TASK_TYPE} | Depth: 4 | Seeds: {SEEDS})...")
t0_cat = time.time()

for fold, (trn_idx, val_idx) in enumerate(skf.split(X_train_cat, y_train)):
    X_tr, y_tr = X_train_cat.iloc[trn_idx], y_train[trn_idx]
    X_va, y_va = X_train_cat.iloc[val_idx], y_train[val_idx]

    fold_val_preds = np.zeros(len(val_idx))
    fold_test_preds = np.zeros(len(X_test_cat))

    for seed in SEEDS:
        p = cat_base_params.copy()
        p["random_seed"] = seed
        cb_clf = CatBoostClassifier(**p)
        cb_clf.fit(X_tr, y_tr, eval_set=(X_va, y_va), cat_features=categorical_features, verbose=False)

        fold_val_preds += cb_clf.predict_proba(X_va)[:, 1] / len(SEEDS)
        fold_test_preds += cb_clf.predict_proba(X_test_cat)[:, 1] / len(SEEDS)

        fold_importance = pd.DataFrame({
            "feature": X_tr.columns,
            "importance": cb_clf.get_feature_importance(),
            "fold": fold + 1
        })
        cat_feature_importances = pd.concat([cat_feature_importances, fold_importance], axis=0)

    cat_oof_probs[val_idx] = fold_val_preds
    cat_test_probs += fold_test_preds / N_SPLITS

    fold_auc = roc_auc_score(y_va, fold_val_preds)
    fold_prauc = average_precision_score(y_va, fold_val_preds)
    cat_fold_aucs.append(fold_auc)
    cat_fold_praucs.append(fold_prauc)
    print(f"CatBoost Fold {fold + 1:02d} | ROC-AUC: {fold_auc:.6f} | PR-AUC: {fold_prauc:.6f} | Best Iter: {cb_clf.get_best_iteration()}")

cat_training_duration = time.time() - t0_cat
cat_overall_auc = roc_auc_score(y_train, cat_oof_probs)
cat_overall_prauc = average_precision_score(y_train, cat_oof_probs)

print("-" * 65)
print(f"Shallow CatBoost {N_SPLITS}-Fold Mean ROC-AUC: {np.mean(cat_fold_aucs):.6f} (+/- {np.std(cat_fold_aucs):.6f})")
print(f"Shallow CatBoost Overall OOF ROC-AUC:  {cat_overall_auc:.6f}")
print(f"Shallow CatBoost Overall OOF PR-AUC:   {cat_overall_prauc:.6f}")
print(f"Training completed in:                 {cat_training_duration:.2f} seconds")
print("-" * 65)


# 7. Model 4: PyTorch Tabular ResNet with Entity Embeddings & In-Memory Checkpoints
---
### Breaking the Axis-Aligned Decision Tree Boundary
All three GBDTs (LightGBM, XGBoost, CatBoost) divide the feature space strictly with orthogonal, axis-aligned hyper-rectangles. In contrast, neural networks learn smooth, continuous non-linear representations ($W x + b$) with continuous gradients.

In Kaggle tabular competitions, combining decision trees with a dedicated tabular neural network provides **maximum decorrelation**, delivering substantial variance reduction when ensembled:
1. **Entity Embeddings**: Maps categorical variables into continuous latent spaces ($d = \min(32, \max(4, (\text{cardinality} + 1) // 2))$).
2. **Batch Normalization & Numerical Standardization**: Stabilizes internal covariance shifts across high-variance financial figures.
3. **Residual Skip Connections**: Preserves clean gradient propagation across multi-layer non-linear transformations.
4. **In-Memory Checkpoint Management**: Optimal model weights are held in RAM (`copy.deepcopy`), avoiding disk latency and eliminating checkpoint clutter.


In [ ]:
# Prepare Categorical and Numerical Matrices for PyTorch ResNet
nn_cat_cols = [c for c in categorical_features if c in X_train.columns]
nn_num_cols = [c for c in X_train.columns if c not in nn_cat_cols]

# Build ordinal encoding for categoricals (0 reserved for unknown)
cat_maps = {}
cat_dims = []
for col in nn_cat_cols:
    uniques = X_train[col].astype(str).unique()
    mapping = {val: i + 1 for i, val in enumerate(uniques)}
    cat_maps[col] = mapping
    num_classes = len(uniques) + 1
    emb_dim = min(32, max(4, (num_classes + 1) // 2))
    cat_dims.append((num_classes, emb_dim))

X_train_cat_arr = np.zeros((len(X_train), len(nn_cat_cols)), dtype=np.int64)
X_test_cat_arr = np.zeros((len(X_test), len(nn_cat_cols)), dtype=np.int64)

for i, col in enumerate(nn_cat_cols):
    X_train_cat_arr[:, i] = X_train[col].astype(str).map(cat_maps[col]).fillna(0).values
    X_test_cat_arr[:, i] = X_test[col].astype(str).map(cat_maps[col]).fillna(0).values

X_train_num_raw = X_train[nn_num_cols].values.astype(np.float32)
X_test_num_raw = X_test[nn_num_cols].values.astype(np.float32)

class TabularDataset(Dataset):
    def __init__(self, x_cat, x_num, y=None):
        self.x_cat = torch.tensor(x_cat, dtype=torch.long)
        self.x_num = torch.tensor(x_num, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32).unsqueeze(1) if y is not None else None

    def __len__(self):
        return len(self.x_num)

    def __getitem__(self, idx):
        if self.y is not None:
            return self.x_cat[idx], self.x_num[idx], self.y[idx]
        return self.x_cat[idx], self.x_num[idx]

class ResBlock(nn.Module):
    def __init__(self, dim, dropout=0.2):
        super().__init__()
        self.block = nn.Sequential(
            nn.Linear(dim, dim),
            nn.BatchNorm1d(dim),
            nn.SiLU(),
            nn.Dropout(dropout),
            nn.Linear(dim, dim),
            nn.BatchNorm1d(dim)
        )
        self.act = nn.SiLU()

    def forward(self, x):
        return self.act(x + self.block(x))

class TabularResNet(nn.Module):
    def __init__(self, cat_dims, num_features, hidden_dim=128, num_blocks=2, dropout=0.2):
        super().__init__()
        self.embeddings = nn.ModuleList([
            nn.Embedding(num_classes, emb_dim) for num_classes, emb_dim in cat_dims
        ])
        total_emb_dim = sum(emb_dim for _, emb_dim in cat_dims)
        self.num_bn = nn.BatchNorm1d(num_features)
        
        in_dim = total_emb_dim + num_features
        self.input_layer = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.SiLU(),
            nn.Dropout(dropout)
        )
        self.blocks = nn.ModuleList([ResBlock(hidden_dim, dropout=dropout) for _ in range(num_blocks)])
        self.head = nn.Linear(hidden_dim, 1)

    def forward(self, x_cat, x_num):
        embedded = [emb(x_cat[:, i]) for i, emb in enumerate(self.embeddings)]
        x_emb = torch.cat(embedded, dim=1) if len(embedded) > 0 else None
        x_num_norm = self.num_bn(x_num)
        x = torch.cat([x_emb, x_num_norm], dim=1) if x_emb is not None else x_num_norm
        x = self.input_layer(x)
        for block in self.blocks:
            x = block(x)
        return self.head(x)


In [ ]:
nn_oof_probs = np.zeros(len(X_train))
nn_test_probs = np.zeros(len(X_test))
nn_fold_aucs = []
nn_fold_praucs = []

NN_BATCH_SIZE = 4096 if IS_KAGGLE else 1024
NN_EPOCHS = 12 if IS_KAGGLE else 4
NN_LR = 3e-3

print(f"Starting PyTorch Tabular ResNet {N_SPLITS}-Fold Cross-Validation (Device: {TORCH_DEVICE} | Epochs: {NN_EPOCHS})...")
t0_nn = time.time()

for fold, (trn_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
    scaler = StandardScaler()
    X_tr_num = scaler.fit_transform(X_train_num_raw[trn_idx])
    X_va_num = scaler.transform(X_train_num_raw[val_idx])
    X_te_num = scaler.transform(X_test_num_raw)

    X_tr_cat, y_tr = X_train_cat_arr[trn_idx], y_train[trn_idx]
    X_va_cat, y_va = X_train_cat_arr[val_idx], y_train[val_idx]

    trn_ds = TabularDataset(X_tr_cat, X_tr_num, y_tr)
    val_ds = TabularDataset(X_va_cat, X_va_num, y_va)
    te_ds = TabularDataset(X_test_cat_arr, X_te_num)

    trn_loader = DataLoader(trn_ds, batch_size=NN_BATCH_SIZE, shuffle=True, drop_last=True)
    val_loader = DataLoader(val_ds, batch_size=NN_BATCH_SIZE * 2, shuffle=False)
    te_loader = DataLoader(te_ds, batch_size=NN_BATCH_SIZE * 2, shuffle=False)

    model = TabularResNet(
        cat_dims=cat_dims,
        num_features=X_tr_num.shape[1],
        hidden_dim=128,
        num_blocks=2,
        dropout=0.20
    ).to(TORCH_DEVICE)

    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=NN_LR, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NN_EPOCHS, eta_min=1e-5)

    best_val_auc = 0.0
    best_val_preds = None
    best_model_weights = None

    for epoch in range(NN_EPOCHS):
        model.train()
        for b_cat, b_num, b_y in trn_loader:
            b_cat, b_num, b_y = b_cat.to(TORCH_DEVICE), b_num.to(TORCH_DEVICE), b_y.to(TORCH_DEVICE)
            optimizer.zero_grad()
            logits = model(b_cat, b_num)
            loss = criterion(logits, b_y)
            loss.backward()
            optimizer.step()

        scheduler.step()

        # Evaluate validation fold
        model.eval()
        epoch_preds = []
        with torch.no_grad():
            for b_cat, b_num, _ in val_loader:
                b_cat, b_num = b_cat.to(TORCH_DEVICE), b_num.to(TORCH_DEVICE)
                preds = torch.sigmoid(model(b_cat, b_num)).cpu().numpy().flatten()
                epoch_preds.extend(preds)

        epoch_val_auc = roc_auc_score(y_va, epoch_preds)
        if epoch_val_auc > best_val_auc:
            best_val_auc = epoch_val_auc
            best_val_preds = np.array(epoch_preds)
            best_model_weights = copy.deepcopy(model.state_dict())

    # Load best checkpoint from memory (zero disk I/O latency)
    model.load_state_dict(best_model_weights)
    model.eval()
    fold_te_preds = []
    with torch.no_grad():
        for b_cat, b_num in te_loader:
            b_cat, b_num = b_cat.to(TORCH_DEVICE), b_num.to(TORCH_DEVICE)
            preds = torch.sigmoid(model(b_cat, b_num)).cpu().numpy().flatten()
            fold_te_preds.extend(preds)

    nn_oof_probs[val_idx] = best_val_preds
    nn_test_probs += np.array(fold_te_preds) / N_SPLITS

    fold_prauc = average_precision_score(y_va, best_val_preds)
    nn_fold_aucs.append(best_val_auc)
    nn_fold_praucs.append(fold_prauc)
    print(f"NN Fold {fold + 1:02d}  | ROC-AUC: {best_val_auc:.6f} | PR-AUC: {fold_prauc:.6f}")

nn_training_duration = time.time() - t0_nn
nn_overall_auc = roc_auc_score(y_train, nn_oof_probs)
nn_overall_prauc = average_precision_score(y_train, nn_oof_probs)

print("-" * 65)
print(f"PyTorch ResNet {N_SPLITS}-Fold Mean ROC-AUC: {np.mean(nn_fold_aucs):.6f} (+/- {np.std(nn_fold_aucs):.6f})")
print(f"PyTorch ResNet Overall OOF ROC-AUC:  {nn_overall_auc:.6f}")
print(f"PyTorch ResNet Overall OOF PR-AUC:   {nn_overall_prauc:.6f}")
print(f"Training completed in:               {nn_training_duration:.2f} seconds")
print("-" * 65)


# 8. Quad-Model Multi-Strategy Ensembling & Stacking Benchmark
---
With four fundamentally diverse architectures (Leaf-wise GBDT, Depth-wise GBDT, Oblivious GBDT, and Continuous ResNet), we evaluate **6 distinct ensembling paradigms**:
1. **Equal 4-Way Linear Blend**: $\frac{1}{4}(P_{\text{LGB}} + P_{\text{XGB}} + P_{\text{CAT}} + P_{\text{NN}})$.
2. **SLSQP Probability Space Optimization**: Convex weighting in bounded probability space.
3. **SLSQP Logit Space Optimization**: Unbounded logit space optimization with sigmoid reconstruction.
4. **4-Way Direct Rank Averaging**: Uniform percentile ranking neutralizing calibration divergence.
5. **Direct Rank-Space Nelder-Mead Optimization**: Simplex optimization directly maximizing empirical ROC-AUC on normalized percentile ranks.
6. **Interaction-Augmented Stacking Meta-Learner**: $L_2$-regularized Logistic Regression meta-model conditioned on model logits, transaction difficulty context, and **multiplicative cross-terms** ($\text{logit}_m \times \text{context}$).


In [ ]:
from scipy.optimize import minimize
from scipy.special import expit, logit
from scipy.stats import rankdata
from sklearn.linear_model import LogisticRegression

EPS = 1e-6
lgb_oof_c = np.clip(lgb_oof_probs, EPS, 1 - EPS)
xgb_oof_c = np.clip(xgb_oof_probs, EPS, 1 - EPS)
cat_oof_c = np.clip(cat_oof_probs, EPS, 1 - EPS)
nn_oof_c  = np.clip(nn_oof_probs, EPS, 1 - EPS)

lgb_test_c = np.clip(lgb_test_probs, EPS, 1 - EPS)
xgb_test_c = np.clip(xgb_test_probs, EPS, 1 - EPS)
cat_test_c = np.clip(cat_test_probs, EPS, 1 - EPS)
nn_test_c  = np.clip(nn_test_probs, EPS, 1 - EPS)

lgb_logits_oof = logit(lgb_oof_c)
xgb_logits_oof = logit(xgb_oof_c)
cat_logits_oof = logit(cat_oof_c)
nn_logits_oof  = logit(nn_oof_c)

lgb_logits_test = logit(lgb_test_c)
xgb_logits_test = logit(xgb_test_c)
cat_logits_test = logit(cat_test_c)
nn_logits_test  = logit(nn_test_c)

# 1. Equal 4-Way Linear Blend
oof_equal = (lgb_oof_probs + xgb_oof_probs + cat_oof_probs + nn_oof_probs) / 4.0
oof_equal_test = (lgb_test_probs + xgb_test_probs + cat_test_probs + nn_test_probs) / 4.0
auc_equal = roc_auc_score(y_train, oof_equal)
prauc_equal = average_precision_score(y_train, oof_equal)

# 2. SLSQP Probability Optimization
def linear_obj(weights):
    w1, w2, w3, w4 = weights
    pred = w1 * lgb_oof_probs + w2 * xgb_oof_probs + w3 * cat_oof_probs + w4 * nn_oof_probs
    return -roc_auc_score(y_train, pred)

init_w = [0.25, 0.25, 0.25, 0.25]
bounds = [(0.0, 1.0)] * 4
constraints = ({"type": "eq", "fun": lambda w: sum(w) - 1.0})
opt_linear = minimize(linear_obj, init_w, method="SLSQP", bounds=bounds, constraints=constraints)
w_lin = opt_linear.x
oof_linear_opt = w_lin[0] * lgb_oof_probs + w_lin[1] * xgb_oof_probs + w_lin[2] * cat_oof_probs + w_lin[3] * nn_oof_probs
oof_linear_opt_test = w_lin[0] * lgb_test_probs + w_lin[1] * xgb_test_probs + w_lin[2] * cat_test_probs + w_lin[3] * nn_test_probs
auc_linear_opt = roc_auc_score(y_train, oof_linear_opt)
prauc_linear_opt = average_precision_score(y_train, oof_linear_opt)

# 3. SLSQP Logit Space Optimization
def logit_obj(weights):
    w1, w2, w3, w4 = weights
    blend_logits = w1 * lgb_logits_oof + w2 * xgb_logits_oof + w3 * cat_logits_oof + w4 * nn_logits_oof
    return -roc_auc_score(y_train, expit(blend_logits))

opt_logit = minimize(logit_obj, init_w, method="SLSQP", bounds=bounds, constraints=constraints)
w_log = opt_logit.x
oof_logit_opt = expit(w_log[0] * lgb_logits_oof + w_log[1] * xgb_logits_oof + w_log[2] * cat_logits_oof + w_log[3] * nn_logits_oof)
oof_logit_opt_test = expit(w_log[0] * lgb_logits_test + w_log[1] * xgb_logits_test + w_log[2] * cat_logits_test + w_log[3] * nn_logits_test)
auc_logit_opt = roc_auc_score(y_train, oof_logit_opt)
prauc_logit_opt = average_precision_score(y_train, oof_logit_opt)

# 4. Pure 3-Way GBDT Rank Averaging (0.94711 Champion Baseline)
N_samples = len(y_train)
lgb_rank = rankdata(lgb_oof_probs) / N_samples
xgb_rank = rankdata(xgb_oof_probs) / N_samples
cat_rank = rankdata(cat_oof_probs) / N_samples
nn_rank  = rankdata(nn_oof_probs) / N_samples

test_lgb_rank = rankdata(lgb_test_probs) / len(lgb_test_probs)
test_xgb_rank = rankdata(xgb_test_probs) / len(xgb_test_probs)
test_cat_rank = rankdata(cat_test_probs) / len(cat_test_probs)
test_nn_rank  = rankdata(nn_test_probs) / len(nn_test_probs)

oof_rank_3way = (lgb_rank + xgb_rank + cat_rank) / 3.0
oof_rank_3way_test = (test_lgb_rank + test_xgb_rank + test_cat_rank) / 3.0
auc_rank_3way = roc_auc_score(y_train, oof_rank_3way)
prauc_rank_3way = average_precision_score(y_train, oof_rank_3way)

# 4B. 4-Way Direct Rank Averaging
oof_rank_4way = (lgb_rank + xgb_rank + cat_rank + nn_rank) / 4.0
oof_rank_4way_test = (test_lgb_rank + test_xgb_rank + test_cat_rank + test_nn_rank) / 4.0
auc_rank_4way = roc_auc_score(y_train, oof_rank_4way)
prauc_rank_4way = average_precision_score(y_train, oof_rank_4way)

# 5. Direct Rank-Space Nelder-Mead Optimization (Seeded from 3-Way Champion)
def rank_nelder_mead_loss(weights):
    w = np.maximum(weights, 0)
    s = w.sum()
    if s == 0:
        return 0.0
    w = w / s
    pred_r = w[0] * lgb_rank + w[1] * xgb_rank + w[2] * cat_rank + w[3] * nn_rank
    return -roc_auc_score(y_train, pred_r)

init_w_nm = [0.34, 0.33, 0.33, 0.00]
opt_nm = minimize(rank_nelder_mead_loss, init_w_nm, method="Nelder-Mead")
w_nm = np.maximum(opt_nm.x, 0)
w_nm = w_nm / w_nm.sum()
oof_rank_nm = w_nm[0] * lgb_rank + w_nm[1] * xgb_rank + w_nm[2] * cat_rank + w_nm[3] * nn_rank
oof_rank_nm_test = w_nm[0] * test_lgb_rank + w_nm[1] * test_xgb_rank + w_nm[2] * test_cat_rank + w_nm[3] * test_nn_rank
auc_rank_nm = roc_auc_score(y_train, oof_rank_nm)
prauc_rank_nm = average_precision_score(y_train, oof_rank_nm)

# 6. Interaction-Augmented 2nd-Level Stacking Meta-Learner
context_cols = ["log_amount_usd", "velocity_score", "merchant_risk_score", "composite_risk_score", "cvv_retry_count"]
scaler_context = StandardScaler()
X_train_context = scaler_context.fit_transform(X_train[context_cols].values)
X_test_context = scaler_context.transform(X_test[context_cols].values)

raw_logits_train = np.column_stack([lgb_logits_oof, xgb_logits_oof, cat_logits_oof, nn_logits_oof])
raw_logits_test  = np.column_stack([lgb_logits_test, xgb_logits_test, cat_logits_test, nn_logits_test])

interactions_train = np.column_stack([
    lgb_logits_oof * X_train_context[:, 0],   # lgb * amount
    cat_logits_oof * X_train_context[:, 4],   # cat * cvv_retry
    xgb_logits_oof * X_train_context[:, 1],   # xgb * velocity
    nn_logits_oof  * X_train_context[:, 3],   # nn  * composite_risk
    lgb_logits_oof * cat_logits_oof,          # tree-oblivious interaction
    lgb_logits_oof * nn_logits_oof            # tree-neural interaction
])

interactions_test = np.column_stack([
    lgb_logits_test * X_test_context[:, 0],
    cat_logits_test * X_test_context[:, 4],
    xgb_logits_test * X_test_context[:, 1],
    nn_logits_test  * X_test_context[:, 3],
    lgb_logits_test * cat_logits_test,
    lgb_logits_test * nn_logits_test
])

X_meta_train = np.column_stack([raw_logits_train, X_train_context, interactions_train])
X_meta_test  = np.column_stack([raw_logits_test,  X_test_context,  interactions_test])

meta_skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=GLOBAL_SEED)
oof_stack = np.zeros(len(y_train))
test_stack = np.zeros(len(X_test))

for m_trn_idx, m_val_idx in meta_skf.split(X_meta_train, y_train):
    m_X_tr, m_y_tr = X_meta_train[m_trn_idx], y_train[m_trn_idx]
    m_X_va, m_y_va = X_meta_train[m_val_idx], y_train[m_val_idx]
    
    meta_clf = LogisticRegression(C=0.05, penalty="l2", solver="lbfgs", max_iter=1000, random_state=GLOBAL_SEED)
    meta_clf.fit(m_X_tr, m_y_tr)
    oof_stack[m_val_idx] = meta_clf.predict_proba(m_X_va)[:, 1]
    test_stack += meta_clf.predict_proba(X_meta_test)[:, 1] / N_SPLITS

auc_stack = roc_auc_score(y_train, oof_stack)
prauc_stack = average_precision_score(y_train, oof_stack)

# Deterministic strategy prediction mapping
strategy_oof_map = {
    "Model 1: Shallow LightGBM (Single)": lgb_oof_probs,
    "Model 2: Shallow XGBoost (Single)": xgb_oof_probs,
    "Model 3: Shallow CatBoost (Single)": cat_oof_probs,
    "Model 4: PyTorch Tabular ResNet (Single)": nn_oof_probs,
    "Ensemble 1: Equal 4-Way Linear Blend": oof_equal,
    "Ensemble 2: SLSQP Probability Blend": oof_linear_opt,
    "Ensemble 3: SLSQP Logit-Space Blend": oof_logit_opt,
    "Ensemble 4A: Pure 3-Way GBDT Rank Average (0.94711 Champion)": oof_rank_3way,
    "Ensemble 4B: 4-Way (+ResNet) Rank Average": oof_rank_4way,
    "Ensemble 5: Rank-Space Nelder-Mead Optimization": oof_rank_nm,
    "Ensemble 6: Interaction-Augmented Stacking Meta-Learner": oof_stack
}

strategy_test_map = {
    "Model 1: Shallow LightGBM (Single)": lgb_test_probs,
    "Model 2: Shallow XGBoost (Single)": xgb_test_probs,
    "Model 3: Shallow CatBoost (Single)": cat_test_probs,
    "Model 4: PyTorch Tabular ResNet (Single)": nn_test_probs,
    "Ensemble 1: Equal 4-Way Linear Blend": oof_equal_test,
    "Ensemble 2: SLSQP Probability Blend": oof_linear_opt_test,
    "Ensemble 3: SLSQP Logit-Space Blend": oof_logit_opt_test,
    "Ensemble 4A: Pure 3-Way GBDT Rank Average (0.94711 Champion)": oof_rank_3way_test,
    "Ensemble 4B: 4-Way (+ResNet) Rank Average": oof_rank_4way_test,
    "Ensemble 5: Rank-Space Nelder-Mead Optimization": oof_rank_nm_test,
    "Ensemble 6: Interaction-Augmented Stacking Meta-Learner": test_stack
}

ensemble_results = pd.DataFrame({
    "Strategy / Architecture": list(strategy_oof_map.keys()),
    "OOF ROC-AUC": [
        lgb_overall_auc,
        xgb_overall_auc,
        cat_overall_auc,
        nn_overall_auc,
        auc_equal,
        auc_linear_opt,
        auc_logit_opt,
        auc_rank_3way,
        auc_rank_4way,
        auc_rank_nm,
        auc_stack
    ],
    "OOF PR-AUC": [
        lgb_overall_prauc,
        xgb_overall_prauc,
        cat_overall_prauc,
        nn_overall_prauc,
        prauc_equal,
        prauc_linear_opt,
        prauc_logit_opt,
        prauc_rank_3way,
        prauc_rank_4way,
        prauc_rank_nm,
        prauc_stack
    ]
}).sort_values(by="OOF ROC-AUC", ascending=False).reset_index(drop=True)

print("=" * 75)
print("QUAD-MODEL ENSEMBLE BENCHMARK RESULTS")
print("=" * 75)
display(ensemble_results)
print("=" * 75)
print(f"SLSQP Probability Weights: [LGB: {w_lin[0]:.3f}, XGB: {w_lin[1]:.3f}, CAT: {w_lin[2]:.3f}, NN: {w_lin[3]:.3f}]")
print(f"SLSQP Logit Weights:       [LGB: {w_log[0]:.3f}, XGB: {w_log[1]:.3f}, CAT: {w_log[2]:.3f}, NN: {w_log[3]:.3f}]")
print(f"Rank Nelder-Mead Weights:  [LGB: {w_nm[0]:.3f}, XGB: {w_nm[1]:.3f}, CAT: {w_nm[2]:.3f}, NN: {w_nm[3]:.3f}]")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))

pred_matrix = pd.DataFrame({
    "LightGBM": lgb_oof_probs,
    "XGBoost": xgb_oof_probs,
    "CatBoost": cat_oof_probs,
    "PyTorch ResNet": nn_oof_probs
})
sns.heatmap(pred_matrix.corr(), annot=True, fmt=".4f", cmap="YlGnBu", cbar=True, ax=axes[0],
            annot_kws={"fontsize": 11, "fontweight": "bold"})
axes[0].set_title("Quad-Model OOF Correlation Matrix", fontsize=12, fontweight="bold")

best_strategy = ensemble_results.iloc[0]["Strategy / Architecture"]
best_auc_score = ensemble_results.iloc[0]["OOF ROC-AUC"]
best_oof_preds = strategy_oof_map.get(best_strategy, oof_rank_3way)

fpr_lgb, tpr_lgb, _ = roc_curve(y_train, lgb_oof_probs)
fpr_xgb, tpr_xgb, _ = roc_curve(y_train, xgb_oof_probs)
fpr_cat, tpr_cat, _ = roc_curve(y_train, cat_oof_probs)
fpr_nn,  tpr_nn,  _ = roc_curve(y_train, nn_oof_probs)
fpr_ens, tpr_ens, _ = roc_curve(y_train, best_oof_preds)

axes[1].plot(fpr_lgb, tpr_lgb, label=f"LightGBM (AUC = {lgb_overall_auc:.5f})", color=PALETTE["neutral"], linewidth=1.5)
axes[1].plot(fpr_xgb, tpr_xgb, label=f"XGBoost  (AUC = {xgb_overall_auc:.5f})", color=PALETTE["accent"], linewidth=1.5)
axes[1].plot(fpr_cat, tpr_cat, label=f"CatBoost (AUC = {cat_overall_auc:.5f})", color=PALETTE["legitimate"], linewidth=1.5)
axes[1].plot(fpr_nn,  tpr_nn,  label=f"ResNet   (AUC = {nn_overall_auc:.5f})", color=PALETTE["dark"], linewidth=1.5)
axes[1].plot(fpr_ens, tpr_ens, label=f"{best_strategy} (AUC = {best_auc_score:.5f})", color=PALETTE["highlight"], linewidth=2.5, linestyle="--")
axes[1].plot([0, 1], [0, 1], "k:", alpha=0.5, label="Chance")

axes[1].set_title("Comparative Out-Of-Fold ROC Curves", fontsize=12, fontweight="bold")
axes[1].set_xlabel("False Positive Rate (1 - Specificity)", fontsize=10)
axes[1].set_ylabel("True Positive Rate (Sensitivity)", fontsize=10)
axes[1].legend(loc="lower right", fontsize=9)
axes[1].grid(True, linestyle="--", alpha=0.6)

plt.suptitle("Quad-Model Architecture Ensembling & Decorrelation Benchmark", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()


# 9. Final Submission & Integrity Verification
---
We generate final test set predictions using the winning ensemble strategy from cross-validation and run defensive integrity assertions:
1. Exact row count match ($299,984$ rows matching `sample_submission.csv`).
2. Zero NaN / null prediction assertion.
3. Strict bounded probability range assertion ($0.0 \le P \le 1.0$).
4. Cross-partition probability calibration density check.
5. Dual-destination export for guaranteed detection by Kaggle's evaluation engine.


In [ ]:
print(f"Generating test predictions using winning strategy: '{best_strategy}'...")

final_test_predictions = strategy_test_map.get(best_strategy, oof_rank_3way_test)

final_submission = pd.DataFrame({
    "id": test_ids,
    "is_fraud": final_test_predictions
})

# Rigorous Defensive Integrity Assertions
assert len(final_submission) == 299984, f"Row count mismatch: expected 299,984, got {len(final_submission)}"
assert not final_submission["is_fraud"].isnull().any(), "Integrity Error: Found NaN values in test predictions!"
assert not np.isinf(final_submission["is_fraud"]).any(), "Integrity Error: Found infinite values in test predictions!"
assert (final_submission["is_fraud"].min() >= 0.0) and (final_submission["is_fraud"].max() <= 1.0), "Integrity Error: Predictions outside [0, 1] bounds!"

# Dual-destination export for guaranteed Kaggle submission detection
output_destinations = ["submission.csv"]
if os.path.exists("/kaggle/working"):
    output_destinations.append("/kaggle/working/submission.csv")

for out_path in output_destinations:
    final_submission.to_csv(out_path, index=False)
    print(f"Verified submission export at: {os.path.abspath(out_path)} (Size: {os.path.getsize(out_path):,} bytes)")

print("=" * 65)
print(f"Winning Strategy:         {best_strategy}")
print(f"Submission Dimensions:    {final_submission.shape[0]:,} rows × {final_submission.shape[1]} columns")
print(f"Min Prediction:           {final_submission['is_fraud'].min():.6f}")
print(f"Mean Prediction:          {final_submission['is_fraud'].mean():.6f}")
print(f"Median Prediction:        {final_submission['is_fraud'].median():.6f}")
print(f"Max Prediction:           {final_submission['is_fraud'].max():.6f}")
print(f"Fraud Ratio (@ p>0.5):    {(final_submission['is_fraud'] > 0.5).mean() * 100:.3f}%")
print("=" * 65)


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(final_submission["is_fraud"], bins=60, color=PALETTE["accent"], kde=True, ax=ax)
ax.set_title("Final Ensemble Test Probability Distribution", fontsize=13, fontweight="bold", pad=12)
ax.set_xlabel("Predicted Probability of Fraud", fontsize=11)
ax.set_ylabel("Frequency", fontsize=11)
ax.set_yscale("log")
ax.grid(True, linestyle="--", alpha=0.6)
plt.tight_layout()
plt.show()
